# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadtalat111/flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Method: Random Forest Classifier (predicting probability, evaluated at Precision@K).
Why it fits the lane: Our goal is to build a "which first?" ranking queue for metadata updates. Following the skills router, we can treat this as a classification problem: predicting whether a high-ranking page belongs in the "Critical CTR" bucket (< 3.0%). We will rank the queue by the model's confidence (probability). I am choosing a Random Forest—a highly reliable tool among standard machine learning frameworks—because it handles non-linear relationships well, doesn't require extreme feature scaling, and provides easily readable feature importances to help us spot leakage during the error analysis.

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_score
import warnings
warnings.filterwarnings('ignore')

# Fixing the random seed for reproducibility as required by the SKILL doc
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Libraries loaded. Random seed fixed to", RANDOM_SEED)


Libraries loaded. Random seed fixed to 42


## 2. Split design

Split Design: Grouped by client_id.
Why this split is honest: The FlyRank data dictionary strictly warns that client_ids are pseudonyms and must be used for grouped splits. If we used a standard random train_test_split, rows from the same client would leak into both the training and validation sets. The model might artificially inflate its score by memorizing a specific client's baseline traffic patterns rather than learning true content signals. Grouping by client_id forces the model to prove it can generalize to entirely unseen clients.

In [5]:
# 1. Load the dataset (using the direct GitHub raw URL for Colab compatibility)
repo_url = 'https://raw.githubusercontent.com/saadtalat111/flyrank/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(repo_url)

# 2. Apply our baseline lane filters (Targeting top-ranking pages)
df_clean = df[df['avg_position'] > 0].copy() # Gotcha: 0 means no data
df_clean = df_clean[df_clean['avg_position'] <= 5].copy() # Only look at highly visible pages

# 3. Create our binary label: 1 if CTR is critical (<3.0%), 0 if it is healthy
df_clean['is_critical_ctr'] = (df_clean['ctr'] < 3.0).astype(int)

# 4. Define features (Ensuring we drop target-derived columns like trend_pct)
# We drop 'ctr' because it's our target. We drop IDs because they are pseudonyms.
features = ['avg_position', 'word_count', 'scroll_rate', 'ai_traffic_pct']
# Fill missing numeric values with 0 (as a baseline assumption for this pass)
X = df_clean[features].fillna(0)
y = df_clean['is_critical_ctr']
groups = df_clean['client_id']

# 5. Execute the Grouped Split (80% train, 20% validation clients)
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=RANDOM_SEED)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

print(f"Total highly visible pages: {len(df_clean)}")
print(f"Training set: {len(X_train)} rows")
print(f"Validation set: {len(X_val)} rows")
print(f"Base rate (Critical CTR) in Validation: {y_val.mean():.1%}")

Total highly visible pages: 3923
Training set: 3036 rows
Validation set: 887 rows
Base rate (Critical CTR) in Validation: 97.1%


## 3. Train + compare vs my baseline

The Comparison: Since our goal is to build a ranked queue of pages to fix, accuracy is the wrong metric (especially with a 97.1% base rate). We care about the top of the queue. We will compare the Baseline rule's Top 50 and Top 100 picks against the Random Forest's Top 50 and Top 100 probabilities using Precision@K.

In [6]:
# 1. Train the Honest Model (Restricting depth to keep it simple/readable)
rf = RandomForestClassifier(random_state=RANDOM_SEED, max_depth=5, n_estimators=100)
rf.fit(X_train, y_train)

# 2. Predict probabilities for the validation set (probability of being class 1: Critical CTR)
val_probs = rf.predict_proba(X_val)[:, 1]

# 3. Rebuild the Baseline on the validation set for a fair fight
val_results = X_val.copy()
val_results['actual_critical_ctr'] = y_val
val_results['rf_prob'] = val_probs

# Pull CTR back in just to calculate the baseline score (The ML model did NOT get to see this)
val_results['ctr'] = df_clean.loc[X_val.index, 'ctr']
# Baseline Score: (6 - avg_position) * (3.0 - ctr)
val_results['baseline_score'] = (6 - val_results['avg_position']) * (3.0 - val_results['ctr'])
val_results.loc[val_results['ctr'] >= 3.0, 'baseline_score'] = 0 # Zero out healthy pages

# 4. Function to evaluate Precision@K
def precision_at_k(df, score_col, k):
    top_k = df.sort_values(by=score_col, ascending=False).head(k)
    return top_k['actual_critical_ctr'].mean()

# 5. Build the Non-Negotiable Comparison Table
k_values = [50, 100, 200]
comparison_data = []
base_rate = y_val.mean()

for k in k_values:
    rf_prec = precision_at_k(val_results, 'rf_prob', k)
    base_prec = precision_at_k(val_results, 'baseline_score', k)
    comparison_data.append({
        'Metric': f'Precision@{k}',
        'Base Rate': f"{base_rate:.1%}",
        'Baseline Rule': f"{base_prec:.1%}",
        'Random Forest': f"{rf_prec:.1%}"
    })

comparison_table = pd.DataFrame(comparison_data)
print("--- Model vs Baseline Comparison (Validation Set) ---")
print(comparison_table.to_string(index=False))

# 6. Extract Feature Importances for Section 4
importances = pd.DataFrame({
    'Feature': features,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)
print("\n--- Feature Importances ---")
print(importances.to_string(index=False))

--- Model vs Baseline Comparison (Validation Set) ---
       Metric Base Rate Baseline Rule Random Forest
 Precision@50     97.1%        100.0%        100.0%
Precision@100     97.1%        100.0%         99.0%
Precision@200     97.1%        100.0%         98.5%

--- Feature Importances ---
       Feature  Importance
    word_count    0.690572
   scroll_rate    0.172582
  avg_position    0.117520
ai_traffic_pct    0.019326


## 4. Errors and interpretation

Feature Interpretation:
The model leaned overwhelmingly on word_count (69% importance), followed by scroll_rate (17%), barely utilizing avg_position (12%). This suggests that in this dataset, content length and engagement proxies were the strongest available predictors of a critical CTR, possibly because very short (or zero-filled) articles suffer the most in search visibility.
Error Analysis:
The most significant finding is that the complex Random Forest lost to the simple Baseline math formula.
The Comparison: At Precision@100 and Precision@200, the Baseline Rule maintained a perfect 100% precision, while the Random Forest dropped to 99.0% and 98.5%.
Why the model is wrong: The Baseline Rule directly mathematically optimized for the worst positions and lowest CTRs. The Random Forest, blinded to the actual CTR (to prevent target leakage), had to infer low click-through rates by looking at proxy metrics like word count and scroll rate. It did an excellent job (98.5% at K=200 is still fantastic given a 97.1% base rate), but it made slight errors because word count is an imperfect proxy for user click intent.
Conclusion: The baseline wins. We do not need a learned model for this specific lane; the deterministic rule is simpler, perfectly transparent, and performs better.

In [7]:
# Show cases where the model was highly confident it was a Critical CTR, but it was actually Healthy (0)
false_positives = val_results[(val_results['rf_prob'] > 0.8) & (val_results['actual_critical_ctr'] == 0)]
print(f"Total highly confident errors (predicted bad CTR, but was actually healthy): {len(false_positives)}")
if len(false_positives) > 0:
    print("\nExample errors:")
    print(false_positives[['avg_position', 'word_count', 'scroll_rate', 'rf_prob', 'ctr']].head(3))

Total highly confident errors (predicted bad CTR, but was actually healthy): 24

Example errors:
      avg_position  word_count  scroll_rate   rf_prob    ctr
265            3.8      1506.0         30.0  0.985148   6.85
634            4.1      2909.0         20.0  0.991437  12.50
1697           0.6      3115.0        150.0  0.906857  14.29


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.